# Multi-league check of rating v0.1

Runs the **same method** on every league in your Oracle's Elixir file, one league at a time (each league is compared only with itself),
and puts the validation numbers side by side.

**Why:** to see whether the method holds up beyond LEC (bigger sample, different leagues) before you publish anything.
This is a *test*, not a ranking: ratings are only meaningful inside one league's baseline.

Needs `rating_core.py` in the same folder as this notebook and the match-data CSV. Press **Run All**.

In [ ]:
# ================= CONFIG =================
FILE = "2026_LoL_esports_match_data_from_OraclesElixir.csv"
LEAGUES = None            # None = every league with enough complete games, or a list like ["LEC", "LCK", "LPL", "LCS"]
MIN_LEAGUE_GAMES = 40     # skip leagues with fewer complete games than this
INCLUDE_PLAYOFFS = False

In [ ]:
import numpy as np
import pandas as pd
import rating_core as rc

df = pd.read_csv(FILE, low_memory=False)
pl = df[df["position"] != "team"]

key_cols = [c for c in ["golddiffat15", "killsat15", "golddiffat25", "damagetotowers", "controlwardsbought", "vspm", "damageshare"] if c in pl.columns]
overview = pd.DataFrame({
    "games": pl.groupby("league")["gameid"].nunique(),
    "pct_complete": pl.groupby("league")["datacompleteness"].apply(lambda s: (s == "complete").mean()) if "datacompleteness" in pl.columns else np.nan,
})
overview = overview.join(pl[key_cols].notna().groupby(pl["league"]).mean().add_prefix("has_"))
print("Leagues in this file (has_* = share of rows where that column is filled):")
print(overview.sort_values("games", ascending=False).round(2).to_string())

In [ ]:
# ---------- run the method league by league ----------
cfg = rc.CFG
if LEAGUES is None:
    LEAGUES = [lg for lg in overview.sort_values("games", ascending=False).index
               if overview.loc[lg, "games"] >= MIN_LEAGUE_GAMES]
print("Leagues to test:", LEAGUES, "\n")

results, frames = {}, {}
for lg in LEAGUES:
    p = rc.load_games(df, lg, include_playoffs=INCLUDE_PLAYOFFS)
    if p["gameid"].nunique() < MIN_LEAGUE_GAMES:
        print(f"{lg}: only {p['gameid'].nunique()} complete games, skipped")
        continue
    p = rc.rate(rc.add_raw_stats(p), cfg)
    if p["r_overall"].notna().mean() < 0.5:
        print(f"{lg}: not enough early/late stats filled in, skipped")
        continue
    if p.attrs.get("skipped_stats"):
        print(f"{lg}: stats skipped (missing/mostly empty): {p.attrs['skipped_stats']}")
    t = rc.player_table(p, cfg)
    res = rc.validate(p, t)
    sens = rc.sensitivity(p, cfg)
    res["weights_stable_min"] = min(v for k, v in sens.items() if k not in ("early only", "late only"))
    res["early_only_corr"] = sens["early only"]
    results[lg], frames[lg] = res, p
    print(f"{lg}: done ({res['games']} games, {res['players']} players)")

In [ ]:
# ---------- side-by-side comparison ----------
summary = pd.DataFrame(results).T
labels = {
    "games": "games", "players": "players",
    "gap_early": "win-loss gap early", "gap_late": "win-loss gap late",
    "corr_gold15_early": "corr early~gold@15", "corr_gold15_overall": "corr overall~gold@15",
    "rel_early": "reliability early", "rel_late": "reliability late", "rel_overall": "reliability overall",
    "rel_within_team": "reliability within team", "corr_rating_team_winrate": "corr rating~team win rate",
    "acc_player_ratings": "predict: ratings", "acc_team_winrate": "predict: team record",
    "weights_stable_min": "weights stability (min rank corr)", "early_only_corr": "early-only rank corr",
}
show = summary[list(labels)].rename(columns=labels)
pd.set_option("display.width", 250)
print(show.round(2).T.to_string())
show.round(3).to_csv("multi_league_validation.csv")
print("\nSaved: multi_league_validation.csv")

In [ ]:
# ---------- all leagues together: biggest sample for the validation tests ----------
# Each league's games were already rated against its OWN league baseline, so combining them keeps leagues comparable
# to themselves. This pooled run is only for validating the method (reliability, prediction), not for ranking players.
pooled = pd.concat(frames.values(), ignore_index=True)
pooled["gameid"] = pooled["gameid"].astype(str)
t_pooled = rc.player_table(pooled, cfg)
pv = rc.validate(pooled, t_pooled)
print(f"POOLED: {pv['games']} games, {pv['players']} players across {len(frames)} leagues")
for k in ["gap_early", "gap_late", "rel_early", "rel_late", "rel_overall", "rel_within_team",
          "corr_rating_team_winrate", "acc_player_ratings", "acc_team_winrate"]:
    print(f"  {labels[k]:28s} {pv[k]:.2f}")

## How to read the table

* **reliability overall** (about 0.8+ is good): do two halves of a player's games rank players alike? Should look similar in every league.
* **reliability within team**: reliability after removing the team's effect. If it is near zero in some league, the rating there is mostly "good team".
* **predict: ratings vs team record**: if ratings beat the team record in some league, that is good news; if they are lower, the ratings add little there.
* **weights stability** close to 1.00 means the exact weights do not matter in that league.
* **If one league looks very different** (for example a league with missing columns), check the "stats skipped" lines above the table first.